# Notebook 03 — LSTM Single-Site
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polar-bear-after-lunch/AareML/blob/main/notebooks/03_lstm_single_site.ipynb)

Trains a Seq2Seq LSTM with Optuna hyperparameter tuning (20 trials) on gauge 2473. Compares default and best-tuned configurations against baselines from notebook 02.

# AareML — 03 LSTM Single Site

**Project:** AareML — Predicting River Water Quality in Swiss Catchments  
**Dataset:** CAMELS-CH-Chem (Nascimento et al., 2025)  
**Reference benchmark:** LakeBeD-US (McAfee et al., 2025)

---

## Architecture

Seq2Seq LSTM matching the LakeBeD-US benchmark design:

```
Encoder LSTM (depth=2)  →  hidden state  →  Decoder LSTM (depth=2)
  input: [lookback, n_feat]                  output: [horizon, n_targets]
```

| Setting | Value |
|---------|-------|
| Lookback | 21 days |
| Forecast horizon | 14 days |
| Targets | DO (mg/L), Temperature (°C) |
| Tuning | Optuna TPE (20 trials) |
| Loss | MSE (scaled space) |
| Optimiser | AdamW + ReduceLROnPlateau |
| Early stopping | patience=12 |
| Metrics | RMSE, MAE, NSE, KGE + 95% bootstrap CIs |


## 0 · Setup


In [1]:
# ── CPU thread optimisation ────────────────────────────────────────────────
# Must run BEFORE importing torch — sets the number of threads PyTorch uses.
# On a 4-core iMac set to 4; on 8-core set to 8. Check with os.cpu_count().
import os
import multiprocessing
N_CPU = multiprocessing.cpu_count()
os.environ['OMP_NUM_THREADS']   = str(N_CPU)
os.environ['MKL_NUM_THREADS']   = str(N_CPU)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_CPU)
print(f'CPU cores detected: {N_CPU} — PyTorch will use all of them')


CPU cores detected: 6 — PyTorch will use all of them


In [2]:
import sys, random, time
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.preprocessing import StandardScaler

from src.config  import *
from src.data    import load_gauge, load_metadata, preprocess, train_val_test_split, make_windows
from src.metrics import rmse_per_step, metrics_table, block_bootstrap_ci, mean_rmse
from src.model   import (
    RiverDataset, Seq2SeqLSTM,
    train_model, predict, get_y_true, save_checkpoint,
)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')


Device: cpu  |  PyTorch: 2.2.2


In [3]:
# ── GPU / DataLoader configuration ────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# On GPU: use 4 workers + pin_memory for faster data transfer to GPU
# On CPU (Mac): use 0 workers (avoids multiprocessing overhead)
NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'DataLoader workers: {NUM_WORKERS}, pin_memory: {PIN_MEMORY}')


Device: cpu
DataLoader workers: 0, pin_memory: False


## 1 · Load and Preprocess


In [4]:
raw  = load_gauge(FOCUS_GAUGE)
data = preprocess(raw)
train, val, test = train_val_test_split(data)

# Training means used for NaN imputation — computed once, reused for all splits
# Compute training means without duplicated index (FEATURES and TARGETS overlap)
train_means = (
    pd.concat([train[FEATURES].mean(), train[TARGETS].mean()])
    .groupby(level=0).first()   # deduplicate temp_sensor / O2C_sensor
)

print(f'Train: {train.index.min().date()} → {train.index.max().date()}  ({len(train):,} days)')
print(f'Val  : {val.index.min().date()}   → {val.index.max().date()}  ({len(val):,} days)')
print(f'Test : {test.index.min().date()}  → {test.index.max().date()}  ({len(test):,} days)')


[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461
Train: 1981-01-01 → 2014-12-31  (12,418 days)
Val  : 2015-01-01   → 2016-12-31  (731 days)
Test : 2017-01-01  → 2020-12-31  (1,461 days)


## 2 · Focus Gauge Metadata


In [5]:
meta = load_metadata()
# Find the gauge row — try common column name variants
id_col = next((c for c in meta.columns if 'gauge' in c.lower() and 'id' in c.lower()), meta.columns[0])
gauge_row = meta[meta[id_col].astype(str) == str(FOCUS_GAUGE)]
if not gauge_row.empty:
    print(f'Focus gauge: {FOCUS_GAUGE}')
    print(gauge_row.T.to_string())
else:
    print(f'Gauge {FOCUS_GAUGE} — id column used: {id_col}')
    print('Column names:', meta.columns.tolist())


Focus gauge: 2473
                                                                                                                 91
gauge_id                                                                                                       2473
sensor_id                                                                                                    2473.0
nawaf_id                                                                                                     1821.0
nawat_id                                                                                                     1821.0
isot_id                                                                                                       NIO03
chirp_id                                                                                                        NaN
gauge_name                                                                                   Diepoldsau, Rietbrücke
water_body_name                                       

## 3 · Feature Scaling

Scalers are fitted **only on training data** and reused across all splits.
NaN imputation uses training-set means — no information from val/test leaks in.


In [6]:
# Impute with training means before fitting scaler
def impute(df, means):
    return df.fillna(means)

feat_scaler = StandardScaler().fit(impute(train[FEATURES], train_means[FEATURES]))
tgt_scaler  = StandardScaler().fit(impute(train[TARGETS],  train_means[TARGETS]))

def scale_split(df):
    X = feat_scaler.transform(impute(df[FEATURES], train_means[FEATURES])).astype(np.float32)
    y = tgt_scaler.transform( impute(df[TARGETS],  train_means[TARGETS])).astype(np.float32)
    return X, y

X_tr_s, y_tr_s = scale_split(train)
X_va_s, y_va_s = scale_split(val)
X_te_s, y_te_s = scale_split(test)

print('Feature means  (train):', dict(zip(FEATURES, feat_scaler.mean_.round(3))))
print('Target  scales (train):', dict(zip(TARGETS,  tgt_scaler.scale_.round(3))))


Feature means  (train): {'temp_sensor': 7.977, 'pH_sensor': 8.104, 'ec_sensor': 294.748, 'O2C_sensor': 11.317}
Target  scales (train): {'O2C_sensor': 0.904, 'temp_sensor': 3.472}


## 4 · Build Windows and Datasets


In [7]:
X_train, y_train, d_train = make_windows(train, train_means)
X_val,   y_val,   d_val   = make_windows(val,   train_means)
X_test,  y_test,  d_test  = make_windows(test,  train_means)


# Build scaled window arrays separately (scaler applied after windowing)
def scale_windows(X_raw, y_raw):
    N, L, F = X_raw.shape
    X_s = feat_scaler.transform(X_raw.reshape(-1, F)).reshape(N, L, F).astype(np.float32)
    Nt, H, T = y_raw.shape
    y_s = tgt_scaler.transform(y_raw.reshape(-1, T)).reshape(Nt, H, T).astype(np.float32)
    return X_s, y_s

Xs_train, ys_train = scale_windows(X_train, y_train)
Xs_val,   ys_val   = scale_windows(X_val,   y_val)
Xs_test,  ys_test  = scale_windows(X_test,  y_test)

ds_train = RiverDataset(Xs_train, ys_train)
ds_val   = RiverDataset(Xs_val,   ys_val)
ds_test  = RiverDataset(Xs_test,  ys_test)

print(f'Train windows: {len(ds_train):,}  |  Val: {len(ds_val):,}  |  Test: {len(ds_test):,}')


[data] make_windows: 12099 windows, X=(12099, 21, 4), y=(12099, 14, 2), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1348 windows, X=(1348, 21, 4), y=(1348, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12099 samples, X=(12099, 21, 4), y=(12099, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1348 samples, X=(1348, 21, 4), y=(1348, 14, 2)
Train windows: 12,099  |  Val: 697  |  Test: 1,348


## 5 · Model Architecture


In [8]:
# Constants derived from config
N_FEAT = len(FEATURES)  # 4
N_TGT  = len(TARGETS)   # 2

# Sanity check: build a model and verify output shape
_m = Seq2SeqLSTM(N_FEAT, N_TGT, hidden=64, n_layers=2, dropout=0.2)
_x = torch.randn(8, LOOKBACK, N_FEAT)
_y = _m(_x)
total_params = sum(p.numel() for p in _m.parameters())
print(f'Output shape : {_y.shape}  (expected [8, {HORIZON}, {N_TGT}])')
print(f'Parameters   : {total_params:,}')
print(_m)


Output shape : torch.Size([8, 14, 2])  (expected [8, 14, 2])
Parameters   : 102,018
Seq2SeqLSTM(
  (encoder): LSTM(4, 64, num_layers=2, batch_first=True, dropout=0.2)
  (decoder): LSTM(2, 64, num_layers=2, batch_first=True, dropout=0.2)
  (drop): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


## 6 · Quick Default Training Run


In [9]:
BATCH_SIZE = 64
dl_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print('Training default model (hidden=64, layers=2, dropout=0.2, lr=1e-3)...')
t0 = time.time()

default_model = Seq2SeqLSTM(N_FEAT, N_TGT, hidden=64, n_layers=2, dropout=0.2)
default_model, hist_default = train_model(
    default_model, dl_train, dl_val,
    lr=1e-3, epochs=100, patience=12, device=DEVICE, verbose=True,
)
print(f'Training time: {time.time()-t0:.1f}s')


Training default model (hidden=64, layers=2, dropout=0.2, lr=1e-3)...
[model] train_model: 102,018 trainable params, device=cpu, epochs=100, patience=12, lr=0.001
  Epoch  10 | train=0.08576  val=0.13302  tf=0.40  lr=1.00e-03
  Early stop at epoch 19  (best val=0.13086)
Training time: 145.3s


## 7 · Training Curve


In [10]:
fig, ax = plt.subplots(figsize=(10, 4))
epochs_x = range(1, len(hist_default['train']) + 1)
ax.plot(epochs_x, hist_default['train'], label='Train loss')
ax.plot(epochs_x, hist_default['val'],   label='Val loss',   linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE loss (scaled targets)')
ax.set_title('LSTM Training Curve — Gauge 2473')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_training_curve.png', bbox_inches='tight')
plt.show()


## 8 · Optuna Hyper-Parameter Tuning

| Parameter | Search space |
|-----------|-------------|
| `hidden` | {32, 64, 128, 256} |
| `n_layers` | 1 – 3 |
| `dropout` | 0.0 – 0.5 |
| `lr` | 1e-4 – 1e-2 (log) |
| `batch_size` | {32, 64, 128} |

Objective: minimum validation MSE (scaled space).


In [11]:
# ── Hyperparameter search space ────────────────────────────────────────────
# Optimisations vs previous version:
# 1. Optuna epochs reduced 60→30 for search (full retraining still uses 150)
# 2. patience reduced 8→5 for search (early-exit bad trials fast)
# 3. Pruning: MedianPruner kills bad trials after epoch 10
# 4. Hidden search space narrowed to [64, 128, 256] — 32 always underfits
# 5. Batch sizes [64, 128] only — 32 is slow and rarely wins
# 6. DataLoaders created once per trial outside the function (if same batch size)
N_TRIALS = 20

import optuna
from optuna.pruners import MedianPruner

def objective(trial):
    hidden     = trial.suggest_categorical('hidden',     [64, 128, 256])
    n_layers   = trial.suggest_categorical('n_layers',   [1, 2])
    dropout    = trial.suggest_float(      'dropout',    0.0, 0.4)
    lr         = trial.suggest_float(      'lr',         1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128])

    dl_tr = DataLoader(ds_train, batch_size=batch_size, shuffle=True,
                       drop_last=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    dl_va = DataLoader(ds_val, batch_size=batch_size, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = Seq2SeqLSTM(N_FEAT, N_TGT, hidden, n_layers, dropout)

    # Reduced epochs+patience for search speed; pruner kills bad trials early
    _, history = train_model(
        model, dl_tr, dl_va,
        lr=lr, epochs=30, patience=5, device=DEVICE, verbose=False,
    )

    val_loss = min(history['val'])

    # Optuna pruning: report intermediate values for early termination
    trial.report(val_loss, step=len(history['val']))
    if trial.should_prune():
        raise optuna.TrialPruned()

    return val_loss

print(f'Running Optuna ({N_TRIALS} trials) with pruning...')
t0 = time.time()

study = optuna.create_study(
    direction='minimize',
    study_name='aareml_gauge2473',
    storage='sqlite:///' + str(Path('../results/optuna_study.db').resolve()),
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f'Finished in {time.time()-t0:.0f}s')
print(f'Completed trials : {len([t for t in study.trials if t.state.name == "COMPLETE"])}')
print(f'Pruned trials    : {len([t for t in study.trials if t.state.name == "PRUNED"])}')

best = study.best_params
print(f'Best val loss : {study.best_value:.6f}')
print(f'Best params   : {best}')


Running Optuna (20 trials) with pruning...


  0%|          | 0/20 [00:00<?, ?it/s]

[model] train_model: 136,450 trainable params, device=cpu, epochs=30, patience=5, lr=0.00012551115172973836
[model] train_model: 535,042 trainable params, device=cpu, epochs=30, patience=5, lr=0.0002049268011541737
[model] train_model: 1,587,714 trainable params, device=cpu, epochs=30, patience=5, lr=0.0005954553793888993
[model] train_model: 136,450 trainable params, device=cpu, epochs=30, patience=5, lr=0.004093813608598782
[model] train_model: 535,042 trainable params, device=cpu, epochs=30, patience=5, lr=0.00011439974749291271
[model] train_model: 35,458 trainable params, device=cpu, epochs=30, patience=5, lr=0.002074566765675252
[model] train_model: 136,450 trainable params, device=cpu, epochs=30, patience=5, lr=0.00045745782054754043
[model] train_model: 1,587,714 trainable params, device=cpu, epochs=30, patience=5, lr=0.0047499747713783975
[model] train_model: 400,642 trainable params, device=cpu, epochs=30, patience=5, lr=0.0004064644058668378
[model] train_model: 102,018 trai

## 9 · Retrain Best Model

Trained on train+val data. Early stopping uses a **held-out shard of val**
(last 20%) that was never part of the training data — avoids the leakage
that occurs when val is inside the training set.


In [12]:
# Split val into training extension (80%) and final early-stop monitor (20%)
n_val = len(ds_val)
n_val_stop = max(30, int(n_val * 0.20))
n_val_train = n_val - n_val_stop

from torch.utils.data import Subset
ds_val_train = Subset(ds_val, range(n_val_train))          # used for training
ds_val_stop  = Subset(ds_val, range(n_val_train, n_val))   # used for early stop only

ds_trainval  = ConcatDataset([ds_train, ds_val_train])
dl_trainval  = DataLoader(ds_trainval, batch_size=best['batch_size'],
                          shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_val_stop  = DataLoader(ds_val_stop,  batch_size=256, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f'Train+val windows: {len(ds_trainval):,}  |  Early-stop monitor: {len(ds_val_stop):,}')

print('Retraining best model...')
t0 = time.time()
best_model = Seq2SeqLSTM(
    N_FEAT, N_TGT,
    hidden=best['hidden'], n_layers=best['n_layers'], dropout=best['dropout'],
)
best_model, hist_best = train_model(
    best_model, dl_trainval, dl_val_stop,
    lr=best['lr'], epochs=150, patience=15, device=DEVICE, verbose=True,
)
print(f'Done in {time.time()-t0:.1f}s')

RESULTS_DIR.mkdir(exist_ok=True)
save_checkpoint(RESULTS_DIR / 'lstm_single_site_best.pt',
                best_model, best, feat_scaler, tgt_scaler)
print('Checkpoint saved → results/lstm_single_site_best.pt')


Train+val windows: 12,657  |  Early-stop monitor: 139
Retraining best model...
[model] train_model: 1,587,714 trainable params, device=cpu, epochs=150, patience=15, lr=0.0017838114892345849
  Epoch  10 | train=0.07123  val=0.19522  tf=0.43  lr=8.92e-04
  Early stop at epoch 17  (best val=0.15794)
Done in 1078.4s
[model] save_checkpoint: saved to /Users/amber/VS Code/polar-bear-after-lunch/AareML/notebooks/../results/lstm_single_site_best.pt (6208.8 KB)
Checkpoint saved → results/lstm_single_site_best.pt


## 10 · Test-Set Evaluation


In [13]:
y_pred_default = predict(default_model, ds_test, tgt_scaler, device=DEVICE)
y_pred_best    = predict(best_model,    ds_test, tgt_scaler, device=DEVICE)

# Ground-truth in original units
y_true_test = get_y_true(ds_test, tgt_scaler)

models_dict = {
    'LSTM (default)': y_pred_default,
    'LSTM (best)':    y_pred_best,
}

# Try to merge with baseline results
lstm_results = metrics_table(models_dict, y_true_test, n_boot=500)
try:
    bl_df = pd.read_csv(RESULTS_DIR / 'baseline_results.csv').rename(columns={'Baseline': 'Model'})
    combined = pd.concat([bl_df, lstm_results], ignore_index=True)
except FileNotFoundError:
    combined = lstm_results
    print('Note: run notebook 02 first to include baseline rows')

print(combined.to_string(index=False))
combined.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
print('\nSaved → results/model_comparison.csv')


[model] predict: 1348 samples, DO range [-2.23, 2.27] mg/L (scaled)
[model] predict: 1348 samples, DO range [-2.20, 2.21] mg/L (scaled)
[metrics] metrics_table for 'LSTM (default)':
         Model    Target   RMSE    MAE   NSE   KGE  RMSE_lo  RMSE_hi  MAE_lo  MAE_hi
LSTM (default) DO (mg/L) 0.2990 0.2363 0.892 0.918   0.2742   0.3229  0.2152  0.2564
LSTM (default) Temp (°C) 1.2525 1.0098 0.884 0.917   1.1518   1.3454  0.9272  1.0929
[metrics] metrics_table for 'LSTM (best)':
      Model    Target   RMSE    MAE   NSE   KGE  RMSE_lo  RMSE_hi  MAE_lo  MAE_hi
LSTM (best) DO (mg/L) 0.3549 0.2791 0.844 0.931   0.3083   0.3980  0.2448  0.3144
LSTM (best) Temp (°C) 1.4334 1.1583 0.847 0.886   1.3025   1.5629  1.0394  1.2800
           Model    Target   RMSE    MAE   NSE   KGE  RMSE_lo  RMSE_hi  MAE_lo  MAE_hi
     Persistence DO (mg/L) 0.3389 0.2655 0.860 0.930   0.3101   0.3669  0.2441  0.2850
     Persistence Temp (°C) 1.3653 1.0736 0.861 0.930   1.2639   1.4722  0.9949  1.1587
     Climatol

## 11 · RMSE by Forecast Horizon


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
steps = np.arange(1, HORIZON + 1)

# Load baselines if available
bl_preds = {}
try:
    bl_df = pd.read_csv(RESULTS_DIR / 'baseline_results.csv')
    print('Baseline results loaded for comparison')
except FileNotFoundError:
    print('Run notebook 02 first for baseline comparison')

model_styles = {
    'LSTM (default)': ('s--', '#BBA8CC'),
    'LSTM (best)':    ('o-',  '#8B4FA8'),
}

for ax_i, (t_i, t) in enumerate(enumerate(TARGETS)):
    ax = axes[ax_i]
    for name, y_pred in models_dict.items():
        marker, color = model_styles[name]
        per = rmse_per_step(y_true_test, y_pred)[:, t_i]
        ax.plot(steps, per, marker, color=color, linewidth=2, label=name, markersize=5)
    ax.set_xlabel('Forecast horizon (days)', fontsize=11)
    ax.set_ylabel('RMSE', fontsize=11)
    ax.set_title(TARGET_LABELS[t], fontsize=12)
    ax.set_xticks(steps)
    ax.legend(fontsize=9)

fig.suptitle('LSTM RMSE by Forecast Horizon — Gauge 2473 (Test Set)',
             fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_rmse_by_horizon.png', bbox_inches='tight')
plt.show()


Baseline results loaded for comparison


In [15]:
# ── I5: Per-horizon RMSE — LSTM vs Ridge baseline ──────────────────────────
import json as _json, pandas as _pd

# Load baseline results for Ridge
baseline_csv = RESULTS_DIR / "baseline_results.csv"
if baseline_csv.exists():
    bl = _pd.read_csv(baseline_csv)
    # baseline_results has columns: Model, Target, RMSE, ... 
    # but we need per-horizon data — load from the horizon CSV if it exists
    horizon_csv = RESULTS_DIR / "baseline_rmse_by_horizon.csv"

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_names = ["DO (mg/L)", "Temp (°C)"]

for ax, tname in zip(axes, target_names):
    # LSTM per-horizon RMSE (best model)
    if 'y_pred_best' in dir() and 'y_true_test' in dir():
        tidx = 0 if "DO" in tname else 1
        rmse_per_h = [
            np.sqrt(np.mean((y_pred_best[:, h, tidx] - y_true_test[:, h, tidx])**2))
            for h in range(HORIZON)
        ]
        ax.plot(range(1, HORIZON+1), rmse_per_h, 'o-', color='#01696F',
                linewidth=2, markersize=4, label='LSTM (best Optuna)')

    ax.set_xlabel("Forecast horizon (days)")
    ax.set_ylabel(f"RMSE ({tname})")
    ax.set_title(f"Per-horizon RMSE — {tname}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("I5: LSTM Per-Horizon RMSE — Gauge 2473 Test Set", fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "03_lstm_rmse_per_horizon_vs_baseline.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_lstm_rmse_per_horizon_vs_baseline.png")


Saved: 03_lstm_rmse_per_horizon_vs_baseline.png


## 12 · Annual RMSE Breakdown

RMSE per calendar year in the test set — shows inter-annual variability
and prevents a single anomalous year from hiding poor generalisation.


In [16]:
years = sorted(set(d_test.year))
annual_rows = []
for yr in years:
    mask = np.array(d_test.year) == yr
    if mask.sum() < 10:
        continue
    for name, y_pred in models_dict.items():
        for t_i, t in enumerate(TARGETS):
            rmse = np.sqrt(((y_true_test[mask,:,t_i] - y_pred[mask,:,t_i])**2).mean())
            annual_rows.append({'Year': yr, 'Model': name,
                                'Target': TARGET_LABELS[t], 'RMSE': round(rmse, 4)})

annual_df = pd.DataFrame(annual_rows)
print(annual_df.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax_i, tgt_label in enumerate([TARGET_LABELS[t] for t in TARGETS]):
    ax = axes[ax_i]
    sub = annual_df[annual_df['Target'] == tgt_label]
    for name, grp in sub.groupby('Model'):
        _, color = model_styles[name]
        ax.plot(grp['Year'], grp['RMSE'], 'o-', color=color, label=name, linewidth=1.8)
    ax.set_xlabel('Year'); ax.set_ylabel('RMSE'); ax.set_title(tgt_label, fontsize=12)
    ax.legend(fontsize=9)

fig.suptitle('Annual RMSE Breakdown — Gauge 2473', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_annual_rmse.png', bbox_inches='tight')
plt.show()


 Year          Model    Target   RMSE
 2017 LSTM (default) DO (mg/L) 0.3246
 2017 LSTM (default) Temp (°C) 1.2451
 2017    LSTM (best) DO (mg/L) 0.4257
 2017    LSTM (best) Temp (°C) 1.5318
 2018 LSTM (default) DO (mg/L) 0.3330
 2018 LSTM (default) Temp (°C) 1.3679
 2018    LSTM (best) DO (mg/L) 0.3811
 2018    LSTM (best) Temp (°C) 1.4838
 2019 LSTM (default) DO (mg/L) 0.2647
 2019 LSTM (default) Temp (°C) 1.1635
 2019    LSTM (best) DO (mg/L) 0.3301
 2019    LSTM (best) Temp (°C) 1.4298
 2020 LSTM (default) DO (mg/L) 0.2879
 2020 LSTM (default) Temp (°C) 1.3062
 2020    LSTM (best) DO (mg/L) 0.3220
 2020    LSTM (best) Temp (°C) 1.4331


## 13 · Example Forecast Plots


In [17]:
def find_nearest_idx(dates, target_date):
    diffs = np.abs((dates - pd.Timestamp(target_date)).days)
    return int(np.argmin(diffs))

winter_idx = find_nearest_idx(d_test, '2018-01-15')
summer_idx = find_nearest_idx(d_test, '2018-07-15')
print(f'Winter window: {d_test[winter_idx].date()}')
print(f'Summer window: {d_test[summer_idx].date()}')

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
steps_arr = np.arange(1, HORIZON + 1)

for row, (label, idx) in enumerate([('Winter 2018', winter_idx), ('Summer 2018', summer_idx)]):
    for col, (t_i, t) in enumerate(enumerate(TARGETS)):
        ax = axes[row][col]
        truth = y_true_test[idx, :, t_i]
        ax.plot(steps_arr, truth, 'k-', linewidth=2, label='Observed')
        for name, y_pred in models_dict.items():
            marker, color = model_styles[name]
            ax.plot(steps_arr, y_pred[idx, :, t_i], marker,
                    color=color, label=name, linewidth=1.5)
        ax.set_title(f'{label} — {TARGET_LABELS[t]}', fontsize=10)
        ax.set_xlabel('Forecast day', fontsize=9)
        ax.set_ylabel(TARGET_LABELS[t], fontsize=9)
        ax.legend(fontsize=8)

fig.suptitle('LSTM Forecast Examples — Gauge 2473', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_example_forecasts.png', bbox_inches='tight')
plt.show()


Winter window: 2018-01-15
Summer window: 2018-07-15


## 14 · Residual Diagnostics


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax_i, (t_i, t) in enumerate(enumerate(TARGETS)):
    ax = axes[ax_i]
    obs  = y_true_test[:, :, t_i].ravel()
    pred = y_pred_best[:, :, t_i].ravel()
    res  = obs - pred
    ax.scatter(pred, res, alpha=0.12, s=4, color='#8B4FA8', rasterized=True)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel(f'Predicted {TARGET_LABELS[t]}', fontsize=10)
    ax.set_ylabel('Residual', fontsize=10)
    bias = res.mean()
    ax.set_title(f'{TARGET_LABELS[t]} — bias={bias:+.3f}', fontsize=11)

plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_lstm_residuals.png', bbox_inches='tight')
plt.show()


## 15 · Optuna Visualisation


In [19]:
try:
    from optuna.visualization.matplotlib import (
        plot_param_importances, plot_optimization_history,
    )
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_param_importances(study, ax=axes[0])
    axes[0].set_title('Parameter Importances', fontsize=11)
    plot_optimization_history(study, ax=axes[1])
    axes[1].set_title('Optimisation History', fontsize=11)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / '03_optuna_summary.png', bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'Optuna matplotlib vis unavailable: {e}')
    top = study.trials_dataframe().sort_values('value').head(10)
    print(top[['number','value','params_hidden','params_n_layers',
               'params_dropout','params_lr','params_batch_size']].to_string(index=False))


Optuna matplotlib vis unavailable: plot_param_importances() got an unexpected keyword argument 'ax'
 number    value  params_hidden  params_n_layers  params_dropout  params_lr  params_batch_size
     12 0.126059            256                2        0.265053   0.001784                128
      7 0.127349            256                2        0.029820   0.004750                 64
     19 0.127683             64                2        0.001322   0.002960                128
     10 0.129836            256                2        0.277907   0.004193                128
      0 0.130564            128                1        0.062398   0.000126                 64
     11 0.130576            256                2        0.305323   0.004691                128
      2 0.130965            256                2        0.146545   0.000595                 64
      3 0.131093            128                1        0.026021   0.004094                 64
      5 0.131771             64              

## 16 · Summary


In [20]:
print('=' * 68)
print('LSTM SINGLE-SITE SUMMARY — Gauge 2473 (Test Set 2017–2020)')
print('=' * 68)
print(f'Best params: {best}')
print()
print(combined.to_string(index=False))
print('=' * 68)
print('LakeBeD-US seq2seq LSTM reference  DO RMSE = 1.40 mg/L')
print('Next → notebook 04_multisite_analysis.ipynb')


LSTM SINGLE-SITE SUMMARY — Gauge 2473 (Test Set 2017–2020)
Best params: {'hidden': 256, 'n_layers': 2, 'dropout': 0.26505333326794106, 'lr': 0.0017838114892345849, 'batch_size': 128}

           Model    Target   RMSE    MAE   NSE   KGE  RMSE_lo  RMSE_hi  MAE_lo  MAE_hi
     Persistence DO (mg/L) 0.3389 0.2655 0.860 0.930   0.3101   0.3669  0.2441  0.2850
     Persistence Temp (°C) 1.3653 1.0736 0.861 0.930   1.2639   1.4722  0.9949  1.1587
     Climatology DO (mg/L) 0.3343 0.2712 0.870 0.853   0.2988   0.3689  0.2438  0.3011
     Climatology Temp (°C) 1.4441 1.1604 0.852 0.884   1.2673   1.6028  1.0157  1.3053
Ridge Regression DO (mg/L) 0.3028 0.2400 0.888 0.908   0.2756   0.3307  0.2181  0.2623
Ridge Regression Temp (°C) 1.2612 1.0192 0.881 0.916   1.1763   1.3557  0.9385  1.1107
  LSTM (default) DO (mg/L) 0.2990 0.2363 0.892 0.918   0.2742   0.3229  0.2152  0.2564
  LSTM (default) Temp (°C) 1.2525 1.0098 0.884 0.917   1.1518   1.3454  0.9272  1.0929
     LSTM (best) DO (mg/L) 0.3549